# N-D-C Synthetic Data Analysis

## Exploring Tholonic Dynamics in Gold Supply Chain

This notebook analyzes simulated N-D-C (Negotiation-Definition-Contribution) data across 4 scenarios:

1. **Baseline**: Healthy, balanced supply chain
2. **Bottleneck**: Phase 6 (Vaulting) severe imbalance
3. **Shock**: Supply disruption at Phase 2 (day 40)
4. **Optimization**: System improving over time

### Learning Objectives:
- Understand D-C balance patterns
- Identify bottlenecks visually
- Observe constraint propagation
- Validate tholonic principles


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 8)
%matplotlib inline

# Load data
data_dir = Path("../../data/processed")
schema_dir = Path("../../schema")

# Load all scenarios
scenarios = {}
for scenario in ['baseline', 'bottleneck', 'shock', 'optimization']:
    file_path = data_dir / f"scenario_{scenario}.csv"
    if file_path.exists():
        scenarios[scenario] = pd.read_csv(file_path)
        scenarios[scenario]['date'] = pd.to_datetime(scenarios[scenario]['date'])
        print(f"✓ Loaded {scenario}: {len(scenarios[scenario])} records")

print(f"\nTotal scenarios: {len(scenarios)}")


## 1. Baseline Scenario Analysis

Healthy supply chain with all phases maintaining D ≈ C balance.


In [ ]:
# Baseline: D vs C Balance by Phase
df_baseline = scenarios['baseline']

# Calculate average D and C per phase
phase_summary = df_baseline.groupby('phase_id').agg({
    'd_value': 'mean',
    'c_value': 'mean',
    'balance_score': 'mean',
    'sustainability_index': 'mean'
}).round(2)

phase_names = ['Prospecting', 'Mining', 'Processing', 'Doré', 
               'Refining', 'Casting', 'Vaulting', 'Exchange']

phase_summary['phase_name'] = phase_names

print("Baseline Scenario - Phase Summary:")
print(phase_summary[['phase_name', 'd_value', 'c_value', 'balance_score', 'sustainability_index']])


In [ ]:
# Visualize D vs C balance
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: D vs C by Phase (Baseline)
ax = axes[0, 0]
x = np.arange(len(phase_names))
width = 0.35

ax.bar(x - width/2, phase_summary['d_value'], width, label='D (Definition)', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, phase_summary['c_value'], width, label='C (Contribution)', color='#3498db', alpha=0.8)

ax.set_xlabel('Supply Chain Phase', fontweight='bold')
ax.set_ylabel('Parameter Value', fontweight='bold')
ax.set_title('Baseline: D vs C Balance by Phase', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(phase_names, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Balance Score by Phase
ax = axes[0, 1]
colors = ['green' if score > 80 else 'yellow' if score > 60 else 'red' 
          for score in phase_summary['balance_score']]
ax.bar(phase_names, phase_summary['balance_score'], color=colors, alpha=0.7)
ax.axhline(y=60, color='orange', linestyle='--', label='Target (60)', linewidth=2)
ax.set_xlabel('Phase', fontweight='bold')
ax.set_ylabel('Balance Score', fontweight='bold')
ax.set_title('Baseline: Balance Score by Phase', fontsize=14, fontweight='bold')
ax.set_xticklabels(phase_names, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Sustainability Index
ax = axes[1, 0]
ax.bar(phase_names, phase_summary['sustainability_index'], color='#2ecc71', alpha=0.7)
ax.set_xlabel('Phase', fontweight='bold')
ax.set_ylabel('Sustainability Index', fontweight='bold')
ax.set_title('Baseline: Sustainability by Phase', fontsize=14, fontweight='bold')
ax.set_xticklabels(phase_names, rotation=45, ha='right')
ax.grid(True, alpha=0.3)

# Plot 4: Time series of system balance
ax = axes[1, 1]
daily_balance = df_baseline.groupby('date')['balance_score'].mean()
ax.plot(daily_balance.index, daily_balance.values, linewidth=2, color='#9b59b6')
ax.set_xlabel('Date', fontweight='bold')
ax.set_ylabel('Average Balance Score', fontweight='bold')
ax.set_title('Baseline: System Balance Over Time', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Baseline shows healthy balance across all phases (avg: {phase_summary['balance_score'].mean():.1f})")


## 2. Bottleneck Scenario Analysis

Phase 6 (Vaulting) has severe D >> C imbalance. Watch stress propagate to adjacent phases.


In [ ]:
# Bottleneck Scenario Analysis
df_bottleneck = scenarios['bottleneck']

bottle_summary = df_bottleneck.groupby('phase_id').agg({
    'd_value': 'mean',
    'c_value': 'mean',
    'balance_score': 'mean',
    'sustainability_index': 'mean'
}).round(2)

bottle_summary['phase_name'] = phase_names

print("Bottleneck Scenario - Phase Summary:")
print(bottle_summary[['phase_name', 'd_value', 'c_value', 'balance_score']])
print(f"\n⚠️ Phase 6 (Vaulting) imbalance: D={bottle_summary.loc[6, 'd_value']}, C={bottle_summary.loc[6, 'c_value']}")
print(f"   Balance score: {bottle_summary.loc[6, 'balance_score']:.1f} (CRITICAL!)")


In [ ]:
# Visualize bottleneck
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Comparison: Baseline vs Bottleneck
ax = axes[0]
x = np.arange(len(phase_names))
width = 0.35

baseline_balance = phase_summary['balance_score'].values
bottleneck_balance = bottle_summary['balance_score'].values

ax.bar(x - width/2, baseline_balance, width, label='Baseline', color='green', alpha=0.7)
ax.bar(x + width/2, bottleneck_balance, width, label='Bottleneck', color='red', alpha=0.7)

ax.axhline(y=60, color='orange', linestyle='--', linewidth=2, label='Target')
ax.set_xlabel('Phase', fontweight='bold')
ax.set_ylabel('Balance Score', fontweight='bold')
ax.set_title('Baseline vs Bottleneck: Balance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(phase_names, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

# Highlight Phase 6 bottleneck
ax.annotate('BOTTLENECK!', xy=(6, bottleneck_balance[6]), 
            xytext=(6, bottleneck_balance[6]-15),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=12, fontweight='bold', color='red')

# Phase 6 time series showing oscillation
ax = axes[1]
phase6_data = df_bottleneck[df_bottleneck['phase_id'] == 6].sort_values('date')
ax.plot(phase6_data['date'], phase6_data['d_value'], label='D (Constraints)', 
        color='red', linewidth=2, alpha=0.8)
ax.plot(phase6_data['date'], phase6_data['c_value'], label='C (Integration)', 
        color='blue', linewidth=2, alpha=0.8)
ax.fill_between(phase6_data['date'], phase6_data['d_value'], phase6_data['c_value'], 
                 alpha=0.2, color='gray', label='Imbalance')

ax.set_xlabel('Date', fontweight='bold')
ax.set_ylabel('Parameter Value', fontweight='bold')
ax.set_title('Phase 6 (Vaulting): D-C Imbalance Over Time', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Shock Scenario Analysis

Supply disruption at Phase 2 (day 40) cascades downstream.


In [ ]:
# Shock Scenario - Watch cascade
df_shock = scenarios['shock']

# Convert date to day number for easier visualization
df_shock['day'] = (df_shock['date'] - df_shock['date'].min()).dt.days

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Plot balance scores over time for each phase
ax = axes[0]
for phase_id in range(8):
    phase_data = df_shock[df_shock['phase_id'] == phase_id].sort_values('day')
    ax.plot(phase_data['day'], phase_data['balance_score'], 
            label=f'Phase {phase_id}: {phase_names[phase_id]}', linewidth=2, alpha=0.8)

ax.axvline(x=40, color='red', linestyle='--', linewidth=3, label='SHOCK at Day 40')
ax.axhline(y=60, color='orange', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Day', fontweight='bold')
ax.set_ylabel('Balance Score', fontweight='bold')
ax.set_title('Shock Scenario: Balance Cascade Across All Phases', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

# Focus on Phase 2 (epicenter)
ax = axes[1]
phase2 = df_shock[df_shock['phase_id'] == 2].sort_values('day')
ax.plot(phase2['day'], phase2['d_value'], label='D (Definition)', color='red', linewidth=2)
ax.plot(phase2['day'], phase2['c_value'], label='C (Contribution)', color='blue', linewidth=2)
ax.axvline(x=40, color='red', linestyle='--', linewidth=3, alpha=0.7, label='Supplier Loss')
ax.fill_between(phase2['day'], phase2['d_value'], phase2['c_value'], 
                 alpha=0.2, color='red' if phase2['d_value'].mean() > phase2['c_value'].mean() else 'blue')
ax.set_xlabel('Day', fontweight='bold')
ax.set_ylabel('Parameter Value', fontweight='bold')
ax.set_title('Phase 2 (Ore Processing): Shock Epicenter', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Downstream cascade (Phases 3-7)
ax = axes[2]
for phase_id in range(3, 8):
    phase_data = df_shock[df_shock['phase_id'] == phase_id].sort_values('day')
    imbalance = abs(phase_data['d_value'] - phase_data['c_value'])
    ax.plot(phase_data['day'], imbalance, label=f'Phase {phase_id}', linewidth=2, alpha=0.8)

ax.axvline(x=40, color='red', linestyle='--', linewidth=3, alpha=0.7, label='Shock')
ax.set_xlabel('Day', fontweight='bold')
ax.set_ylabel('|D - C| Imbalance', fontweight='bold')
ax.set_title('Downstream Cascade: Imbalance Propagation (Phases 3-7)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\\n⚠️ Observe: Shock at Phase 2 (day 40) → Downstream imbalances build over ~15 days")


## 4. Summary: Tholonic Principles Validated

Key observations from synthetic data:


In [ ]:
# Summary statistics across all scenarios
summary_stats = []

for name, df in scenarios.items():
    stats = {
        'Scenario': name.capitalize(),
        'Avg Balance': df['balance_score'].mean(),
        'Min Balance': df['balance_score'].min(),
        'Avg Sustainability': df['sustainability_index'].mean(),
        'System Health': 'Excellent' if df['balance_score'].mean() > 80 else 
                        'Good' if df['balance_score'].mean() > 70 else
                        'Fair' if df['balance_score'].mean() > 60 else 'Poor'
    }
    summary_stats.append(stats)

summary_df = pd.DataFrame(summary_stats)
print("="*70)
print("SCENARIO COMPARISON")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)

print("\n✅ Key Tholonic Principles Validated:")
print("  1. Balance (D ≈ C) → High Sustainability")
print("  2. Imbalance (D ≠ C) → Energy Cost Increases")
print("  3. Bottlenecks Propagate Through Adjacent Phases")
print("  4. Shocks Create Cascading Imbalances")
print("\n📊 Synthetic data successfully demonstrates N-D-C dynamics!")
